### Loading a model and inference Example

In [1]:
## Manual way

import mlflow
import numpy as np
from mlflow.models import Model
from mlflow.tracking import MlflowClient
from mlflow.models import infer_signature
import conda

print(mlflow.__version__)

# ===== MLflow tracking =====
TRACKING_URI = "http://sunrise-mlflow-tracking.mlflow.svc.cluster.local:5080"
EXPERIMENT_NAME = "k8s-cpu-forecasting"
REGISTERED_MODEL_NAME = "cpu-pct"     # model registry name (optional but nice)

mlflow.set_tracking_uri(TRACKING_URI)
mlflow.set_experiment(EXPERIMENT_NAME)
ml_client = MlflowClient()

print("MLflow:", TRACKING_URI, "Experiment:", EXPERIMENT_NAME)

model_uri = 'runs:/c9eee3c98eee4309a0a602339d1ea612/2025-09-08-11:16:55-cpu-node-1-pct-model'
model = mlflow.pyfunc.load_model(model_uri)

# Use example input shape from MLflow UI
input_data = np.array([[
    [0.19565217],
    [0.22826087],
    [0.25],
    [0.20652173],
    [0.22826087]
]],dtype=np.float32)   # shape (1, 5, 1)

result = model.predict(input_data)
print(result)

2.7.1
MLflow: http://sunrise-mlflow-tracking.mlflow.svc.cluster.local:5080 Experiment: k8s-cpu-forecasting


/opt/conda/lib/python3.11/site-packages/pydantic/_internal/_config.py:373: UserWarning: Valid config keys have changed in V2:
* 'schema_extra' has been renamed to 'json_schema_extra'
  warnings.warn(message, UserWarning)


2025/09/23 09:34:15 WARNING mlflow.pyfunc: Detected one or more mismatches between the model's dependencies and the current Python environment:
 - mlflow (current: 2.7.1, required: mlflow==2.21.3)
 - cloudpickle (current: 2.2.1, required: cloudpickle==3.1.1)
To fix the mismatches, call `mlflow.pyfunc.get_model_dependencies(model_uri)` to fetch the model's environment and install dependencies using the resulting environment file.


AttributeError: Can't get attribute '_class_setstate' on <module 'cloudpickle.cloudpickle' from '/opt/conda/lib/python3.11/site-packages/cloudpickle/cloudpickle.py'>

### search  last Runs

In [2]:
from mlflow.tracking import MlflowClient

mlflow.set_tracking_uri(TRACKING_URI)
client = MlflowClient()

# Find the experiment ID
experiment = client.get_experiment_by_name("k8s-cpu-forecasting")
print("Experiment ID:", experiment.experiment_id)

# List the last few finished runs
runs = client.search_runs(
    experiment_ids=[experiment.experiment_id],
    filter_string="attributes.status = 'FINISHED'",
    order_by=["attributes.start_time DESC"],
    max_results=5
)
for r in runs:
    print(f"✔ run_id: {r.info.run_id}   |   status: {r.info.status}   |   start: {r.info.start_time}")


Experiment ID: 27
✔ run_id: c053e72ffe854bc5998fdeba217525c8   |   status: FINISHED   |   start: 1758546571613
✔ run_id: 3c7a68861bdc499686120b49bb73e3aa   |   status: FINISHED   |   start: 1758546566020
✔ run_id: b0d712691700472cb4fa6379c18479ba   |   status: FINISHED   |   start: 1758546541471
✔ run_id: 066c448b6a624e6f871267016e15dabf   |   status: FINISHED   |   start: 1758546516705
✔ run_id: 56baa4913018484b96898c902a53009f   |   status: FINISHED   |   start: 1758546510610


In [3]:
# Loading Artifacts  Example!

In [4]:
import mlflow
from mlflow.artifacts import load_text

# --- CONFIG ---
TRACKING_URI = "http://sunrise-mlflow-tracking.mlflow.svc.cluster.local:5080"
RUN_ID = "cca14992961e480fb5991661fd6f9184"   # ✅ from the image
REQUIREMENTS_PATH = "model/requirements.txt"
# ----------------

mlflow.set_tracking_uri(TRACKING_URI)

# ✅ Load the requirements.txt content
content = load_text(f"runs:/{RUN_ID}/{REQUIREMENTS_PATH}")

# ✅ Parse lines into a clean list
requirements = [line.strip() for line in content.splitlines() if line.strip() and not line.startswith("#")]

# ✅ Print the parsed requirements
print("📦 Parsed requirements:")
for req in requirements:
    print(" -", req)


📦 Parsed requirements:
 - mlflow==2.10.2
 - cloudpickle==3.1.1
 - numpy==1.26.4
 - packaging==23.2
 - pandas==2.3.2
 - pyyaml==6.0.2
 - torch==2.2.1
 - tqdm==4.67.1


### List Experiments

In [5]:
import mlflow
from mlflow.tracking import MlflowClient


TRACKING_URI = "http://sunrise-mlflow-tracking.mlflow.svc.cluster.local:5080"

mlflow.set_tracking_uri(TRACKING_URI)
client = MlflowClient()

# List experiments (active only, change if needed)
experiments = client.search_experiments()

print("MLflow Experiments:")
for exp in experiments:
    print(f" - ID: {exp.experiment_id:<4} | Name: {exp.name} | Lifecycle: {exp.lifecycle_stage}")


MLflow Experiments:
 - ID: 27   | Name: k8s-cpu-forecasting | Lifecycle: active
 - ID: 24   | Name: CPU_Forecasting_Finetuning | Lifecycle: active
 - ID: 23   | Name: CPU_Forecasting_Experiment | Lifecycle: active
 - ID: 21   | Name: AutoAnomalyDetections | Lifecycle: active
 - ID: 20   | Name: AutoMLRegressions | Lifecycle: active
 - ID: 19   | Name: AutoMLClassifications | Lifecycle: active
 - ID: 9    | Name: XGBoost in Breast Cancer dataset-mlflow-icom bucket | Lifecycle: active
 - ID: 2    | Name: XGBoost in Breast Cancer dataset | Lifecycle: active
 - ID: 1    | Name: Ollama | Lifecycle: active
 - ID: 0    | Name: Default | Lifecycle: active
